# Bivariate & Multivariate Analysis with Seaborn

Where earlier EDA notebooks looked at one column at a time, this one is organized around **relationships between columns** — picking the right plot depends on whether each variable is numerical or categorical. Four datasets are used: `tips`, `titanic`, `flights`, and `iris`.

## Setup — Loading the Datasets

`seaborn` ships with a few small built-in datasets (`tips`, `flights`, `iris`) that load directly with `sns.load_dataset()` — handy for practice without needing your own CSV. `titanic` here is still read from the local `train.csv`, as in the earlier notebooks.

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
import matplotlib.pyplot as plt   # needed below for plt.legend() and clustermap/heatmap sizing


In [3]:
tips = sns.load_dataset('tips')

In [4]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
titanic = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
flights = sns.load_dataset('flights')

In [ ]:
flights.head()

In [ ]:
iris = sns.load_dataset('iris')

In [ ]:
iris

## 1. Scatterplot (Numerical - Numerical)

A scatterplot is the standard tool for two **numerical** columns — each point is one row, positioned by its two values. Extra keyword arguments fold in more dimensions without needing a 3D plot:
- `hue=` colors points by a categorical column
- `style=` changes the marker shape by a categorical column
- `size=` scales point size by a numerical (or categorical) column

### ⚠️ Two Issues With the Cell Below

1. **Bug:** `hue=df['sex']` references `df`, which was never defined in this notebook — this raises `NameError`. It should be `tips['sex']` (`tips` is the DataFrame actually loaded above).
2. **Deprecation:** passing `total_bill`/`tip` positionally is no longer accepted on seaborn 0.12+ — they need to be passed as `x=`/`y=` keywords.

In [ ]:
sns.scatterplot(tips['total_bill'],tips['tip'],hue=df['sex'],style=df['smoker'],size=df['size'])

In [ ]:
sns.scatterplot(
    x='total_bill', y='tip', hue='sex', style='smoker', size='size',
    data=tips
)


## 2. Bar Plot (Numerical - Categorical)

In [ ]:
titanic.head()

A barplot compares a **numerical** column's mean **across categories** — here, average age by passenger class, split further by sex via `hue`. The same positional-argument deprecation applies as above.

In [ ]:
sns.barplot(titanic['Pclass'],titanic['Age'],hue=titanic['Sex'])

In [ ]:
sns.barplot(x='Pclass', y='Age', hue='Sex', data=titanic)


## 3. Box Plot (Numerical - Categorical)

A boxplot shows the full **distribution** (median, quartiles, outliers) of a numerical column across categories, rather than just the mean — more informative than a barplot when the spread or presence of outliers matters, not just the average.

In [ ]:
sns.boxplot(titanic['Sex'],titanic['Age'],hue=titanic['Survived'])

In [ ]:
sns.boxplot(x='Sex', y='Age', hue='Survived', data=titanic)


## 4. Distplot (Numerical - Categorical)

The intent here is to compare the **age distribution** of passengers who died (`Survived == 0`) against those who survived (`Survived == 1`), overlaid on the same axes — `hist=False` meant "just draw the smooth density curve, skip the histogram bars."

### ⚠️ Important: `sns.distplot()` Is Removed

As covered in the earlier visualization notebook, `distplot()` no longer exists in current seaborn. Since this cell specifically used `hist=False` (density curve only, no bars), the direct replacement is `sns.kdeplot()` — calling it twice on the same axes overlays both groups, exactly as the original intended.

In [ ]:
sns.distplot(titanic[titanic['Survived']==0]['Age'],hist=False)
sns.distplot(titanic[titanic['Survived']==1]['Age'],hist=False)

In [ ]:
sns.kdeplot(titanic[titanic['Survived'] == 0]['Age'], label='Did not survive')
sns.kdeplot(titanic[titanic['Survived'] == 1]['Age'], label='Survived')
plt.legend()


## 5. HeatMap (Categorical - Categorical)

In [ ]:
titanic.head(3)

For two **categorical** columns, `pd.crosstab()` first builds a table of counts for every combination (here, class × survival), and `sns.heatmap()` renders that table as color-coded cells — patterns jump out faster than reading raw numbers. Passing a single DataFrame positionally is fine for `heatmap()` — it's one of the seaborn functions that never had the positional-argument deprecation.

In [ ]:
sns.heatmap(pd.crosstab(titanic['Pclass'],titanic['Survived']))

### ⚠️ `.groupby().mean()` Also Needs `numeric_only=True` on Modern Pandas

Same issue flagged in the earlier EDA notebook: `titanic` has non-numeric columns (`Name`, `Sex`, `Ticket`, ...). Since pandas 2.0, `.mean()` on a groupby object raises `TypeError` if it hits a non-numeric column, unless `numeric_only=True` is passed — or, cleaner here, just select the one column you actually want averaged.

In [ ]:
(titanic.groupby('Embarked').mean()['Survived']*100)

In [ ]:
# Fixed: select only the column being averaged, avoiding the non-numeric columns entirely
titanic.groupby('Embarked')['Survived'].mean() * 100


### Adding a Heatmap's Numbers as Text — `annot=True`

A plain heatmap shows color only; `annot=True` prints the actual value inside each cell too — usually worth adding by default, since color alone can be hard to read precisely.

In [ ]:
sns.heatmap(pd.crosstab(titanic['Pclass'], titanic['Survived']), annot=True, fmt='d', cmap='Blues')


## 6. ClusterMap (Categorical - Categorical)

A clustermap is a heatmap with an extra step: rows and columns are automatically **reordered and grouped** by similarity (shown as the dendrogram trees on the edges), which can reveal structure a plain heatmap's fixed row/column order would hide.

In [ ]:
sns.clustermap(pd.crosstab(titanic['Parch'],titanic['Survived']))

## 7. Pairplot

In [ ]:
iris.head()

`sns.pairplot()` is a fast way to see **every pairwise relationship at once**: it plots a scatterplot for every pair of numerical columns, with each column's own distribution along the diagonal. `hue='species'` colors every point by the category, making it easy to see which numeric features actually separate the three iris species.

In [ ]:
sns.pairplot(iris,hue='species')

## 8. Lineplot (Numerical - Numerical)

A lineplot connects points in **order along the x-axis** — the right choice when x represents something sequential, like time, rather than an unordered category (which is what a bar or scatter plot suits better).

In [ ]:
flights.head()

### ⚠️ `.groupby().sum()` Can Break on Modern Pandas Too

`flights` includes a `month` column stored as a **categorical** dtype. Summing a categorical column isn't a meaningful operation, and pandas 2.0 raises `TypeError` for it by default (rather than silently skipping it, as older pandas did). Since the goal here is really just "total passengers per year," selecting that one column before summing sidesteps the issue entirely — and is arguably clearer code anyway.

In [ ]:
new = flights.groupby('year').sum().reset_index()

In [ ]:
# Fixed: select the numeric column of interest first, rather than summing the whole DataFrame
new = flights.groupby('year')['passengers'].sum().reset_index()
new.head()


Same positional-argument deprecation as the earlier plots — `x=`/`y=` keywords are required on current seaborn.

In [ ]:
sns.lineplot(new['year'],new['passengers'])

In [ ]:
sns.lineplot(x='year', y='passengers', data=new)


In [ ]:
flights

`pivot_table()` reshapes `flights` from long format (one row per year-month) into a grid — months as rows, years as columns, passenger counts as values — which is exactly the 2D shape `clustermap()` (and `heatmap()`) need as input.

In [ ]:
sns.clustermap(flights.pivot_table(values='passengers',index='month',columns='year'))

## 9. Jointplot — Scatter + Marginal Distributions

`sns.jointplot()` combines a scatterplot with each variable's own distribution along the top and right margins — one plot answers both "how are these two related?" and "what does each one look like on its own?". `kind='reg'` adds a regression line on top of the scatter.

In [ ]:
sns.jointplot(x='total_bill', y='tip', data=tips, hue='sex')


## 10. Regplot / Lmplot — Scatter With a Trend Line

`sns.regplot()` is a scatterplot with a fitted linear regression line (and confidence band) overlaid — useful for eyeballing whether two numerical columns have a roughly linear relationship, and how strong it looks.

In [ ]:
sns.regplot(x='total_bill', y='tip', data=tips)


## 11. Violin Plot — Boxplot + Distribution Shape

`sns.violinplot()` shows the same quartile/median information as a boxplot, but mirrors the distribution's actual shape (like a sideways KDE) around it — useful when a distribution is multi-modal (two humps) and a boxplot alone would hide that.

In [ ]:
sns.violinplot(x='Sex', y='Age', hue='Survived', data=titanic, split=True)


## Summary — Choosing a Plot by Variable Types

| Relationship | Plot(s) | Method |
|---|---|---|
| Numerical – Numerical | Scatter, Line, Regression, Joint | `scatterplot`, `lineplot`, `regplot`, `jointplot` |
| Numerical – Categorical | Bar (mean), Box (spread), Violin (shape), KDE (density) | `barplot`, `boxplot`, `violinplot`, `kdeplot` |
| Categorical – Categorical | Heatmap, Clustermap (on a crosstab) | `heatmap(pd.crosstab(...))`, `clustermap(...)` |
| Many numerical columns at once | Pairplot | `pairplot(df, hue=...)` |

### Gotchas on current seaborn / pandas versions
- Pass plot variables as **keywords** (`x=`, `y=`, `hue=`), not positionally — the old `sns.scatterplot(a, b)` style now errors.
- `sns.distplot()` is **removed** — use `sns.kdeplot()` (density only) or `sns.histplot(kde=True)` (bars + density).
- `.groupby().mean()` / `.sum()` on a DataFrame with non-numeric or categorical columns needs `numeric_only=True`, or (cleaner) select just the numeric column you actually want aggregated before calling `.mean()`/`.sum()`.